# 向量与空间：三种视角、linear combination 与 basis

**用途**：topic 1.6.5 第一组的引导式练习，与 ELI5 页 `web/learn/06-machine-learning-foundations/linear-algebra.html` 逐步对应。每一步四段：**学**（说明与公式）→ **玩**（运行演示）→ **做**（在 `vectors.py` 里手写实现）→ **测**（验证单元格）。

**前提**：Python 3 标准库；`numpy` 只在最后的对拍中使用，缺失时跳过对拍。实现只写在同目录的 `vectors.py`（C5：本人手写，不得引入 numpy 替代）。本 notebook 只讲、只演示、只验证，不含答案。

**运行方式**：从上到下依次执行。「测」单元格在骨架状态下失败是预期结果；每写完一步的函数，重新执行该步的「做」与「测」。交付前用 Restart Kernel and Run All Cells 确认从头能跑通。

来源：3Blue1Brown《Essence of Linear Algebra》第 1–2 集；知识正文《02 - AI - 向量与空间：三种视角、线性组合与基》（私有知识库）。

In [ ]:
# 环境准备：让 notebook 能导入同目录的 vectors.py 与 playground/ 下的模块
import importlib
import sys
from pathlib import Path

HERE = Path.cwd()
for folder in (HERE, HERE / "playground"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import vectors
from display_helpers import color_swatches, report_implemented
from svgplot import Figure


def status(**checks):
    """重新加载 vectors.py 后，报告 checks 里的函数是否已实现；checks 的值是示例参数。"""
    importlib.reload(vectors)
    report_implemented(vectors, checks)

## 概述

### 背景

语言模型内部的数据与 parameter（参数）都以 vector 与 matrix（矩阵）存储：token 的 embedding 是 vector，attention 的输出是 value vector 的 linear combination，LoRA 的权重更新落在少数几个 vector 的 span 内。Vector 的两种运算以及由此定义的 span 与 basis，是后续 matrix、determinant（行列式）与 eigenvector（特征向量）的前提。

### 定义与示例

**Vector** 是一类数学对象，其上定义了两种运算——两个 vector 相加，以及 vector 乘以一个数（scalar，标量）——且运算结果仍是 vector。几何上它是从原点出发的箭头，由长度与方向决定；数值上它是有序的数字列表，顺序与长度均有意义。

示例采用屏幕颜色：一种颜色由红、绿、蓝三个分量表示，是一个三维 vector。

1. **表示**：红色为 (255, 0, 0)，黄色为 (255, 255, 0)。
2. **数乘**：红色乘以 0.5，每个分量减半，得到 (127.5, 0, 0)。
3. **相加**：0.5 倍红色与 0.5 倍黄色逐分量相加，得到 (255, 127.5, 0)，屏幕上显示为橙色。缩放后相加的结果称为 linear combination。
4. **Span**：只用红与绿，蓝分量始终为 0，能配出的颜色只占全部颜色中的一个平面。
5. **Basis**：红、绿、蓝任一种都配不出另外两种（linearly independent），三者又能配出全部颜色，因此构成颜色空间的一个 basis，(R, G, B) 即坐标。

公式：$v + w = [v_1 + w_1,\ v_2 + w_2,\ v_3 + w_3]$，$c \cdot v = [c v_1,\ c v_2,\ c v_3]$；linear combination 为 $a \cdot v + b \cdot w$。

In [ ]:
# 玩：0.5·红 + 0.5·黄。逐分量计算（此处不依赖 vectors.py，因为它还没实现）
red = (255, 0, 0)
yellow = (255, 255, 0)
# zip 把两个向量按分量配对，每对分量各乘 0.5 再相加
orange = tuple(0.5 * r + 0.5 * y for r, y in zip(red, yellow))
print("0.5·红 + 0.5·黄 =", orange)
color_swatches((red, "红 × 0.5"), "+", (yellow, "黄 × 0.5"), "=", (orange, "橙"))

### 起源与发展

Vector 一词来自 Hamilton 1843 年的 quaternion。Grassmann 1844 年的《扩张论》已定义 linear independence、span 与维数，但长期无人理解；Peano 1888 年给出 vector space 的公理化定义，1920 年前后由 Banach、Noether 等人各自重新发现；Halmos 1942 年的教材确立了以 vector space 为起点的现代讲法。详见知识正文《线性代数的起源与发展》。

## 逐步推演

推演在平面上进行，使用 v = [3, 1] 与 w = [1, 2]，步骤 4扩展到三维。本例说明以下要点：

1. 坐标是对 basis vector 的缩放。
2. Linear combination 的几何意义是首尾相接。
3. Span 由 vector 是否共线决定；basis 要求 linearly independent 且 span 整个空间。

### 步骤 1 · 坐标即缩放 basis vector

**学**。î 与 ĵ 是长度为 1、分别指向 x 与 y 方向的 basis vector。每个坐标是一个 scalar，缩放对应的 basis vector；两段缩放后的箭头首尾相接，终点即该 vector。

$$(3, -2) = 3 \cdot (1, 0) + (-2) \cdot (0, 1) = (3, 0) + (0, -2)$$

In [ ]:
# 玩：坐标 (3, −2) 是怎样由 basis vector 缩放再首尾相接得到的
fig = Figure(xlim=(-1, 5), ylim=(-3, 2), title="(3, −2) = 3·î + (−2)·ĵ")
fig.arrow((1, 0), label="î", color=Figure.BLUE, width=3)   # 两个单位向量
fig.arrow((0, 1), label="ĵ", color=Figure.BLUE, width=3)
# 第一段：3 倍的 î；第二段从它的尖端接上 −2 倍的 ĵ；最后一条是合成向量
tip = fig.arrow((3, 0), label="3·î", color=Figure.GREY, dashed=True, label_pos="mid", side=1)
fig.arrow((0, -2), tail=tip, label="(−2)·ĵ", color=Figure.GREY, dashed=True, label_pos="mid", side=-1)
fig.arrow((3, -2), label="(3, −2)")
fig.show()

**做**。在 `vectors.py` 中实现：

- `scale(c, v)`：每个分量乘以 scalar c。
- `add(v, w)`：按分量相加。

要求只用列表与标量运算，不 import numpy。

In [ ]:
status(scale=(2, [1, 0]), add=([1, 0], [0, 1]))

In [ ]:
# 测：本步的数值。每个验证单元格都重新 import，才能拿到 reload 之后的新函数
from vectors import add, scale

assert add(scale(3, [1, 0]), scale(-2, [0, 1])) == [3, -2], "3·î + (−2)·ĵ 应为 [3, −2]"
assert add([1, 2], [3, -1]) == [4, 1]
assert scale(-1.5, [2, -4]) == [-3, 6]
print("步骤 1 通过")

### 步骤 2 · Linear combination 首尾相接

**学**。把 v 拉长到 2 倍，再把 w 平移到它的尖端，从原点到 w 尖端的箭头即 linear combination 的结果。数值上先数乘再逐分量相加。

$$a \cdot v + b \cdot w：\quad 2 \cdot (3, 1) + 1 \cdot (1, 2) = (6, 2) + (1, 2) = (7, 4)$$

系数 (2, 1) 由方程 $3a + b = 7$、$a + 2b = 4$ 解出。

In [ ]:
# 玩：首尾相接
# 黑色：v、w 本身；灰色虚线：2·v，以及接在它尖端的 w；蓝色：结果
V, W, T = (3, 1), (1, 2), (7, 4)
fig = Figure(xlim=(-1, 8), ylim=(-1, 5), title="2·v + 1·w = (7, 4)")
fig.arrow(V, label="v = (3, 1)", label_pos="mid", side=1)
fig.arrow(W, label="w = (1, 2)")
two_v = (2 * V[0], 2 * V[1])
tip = fig.arrow(two_v, label="2·v", color=Figure.GREY, dashed=True, label_pos="mid", side=-1)
fig.arrow(W, tail=tip, label="1·w", color=Figure.GREY, dashed=True, label_pos="mid", side=1)
fig.arrow(T, label="(7, 4)", color=Figure.BLUE, width=3)
fig.show()

In [ ]:
# 玩：命中目标。改动 a、b 再执行，看落点离 T 还差多少
# 打中时的 (a, b) 就是 T 在基 {v, w} 下的坐标
a, b = 1, 1
P = (a * V[0] + b * W[0], a * V[1] + b * W[1])
print(f"a = {a}, b = {b} → 落点 {P}，距目标 {T} 还差 {(T[0] - P[0], T[1] - P[1])}")
fig = Figure(xlim=(-1, 8), ylim=(-1, 5), title=f"a = {a}, b = {b}")
a_v = (a * V[0], a * V[1])
b_w = (b * W[0], b * W[1])
tip = fig.arrow(a_v, label=f"{a}·v", color=Figure.GREY, dashed=True, label_pos="mid", side=-1)
fig.arrow(b_w, tail=tip, label=f"{b}·w", color=Figure.GREY, dashed=True, label_pos="mid", side=1)
fig.point(P, label=f"落点 {P}", color=Figure.BLUE, r=5)
fig.point(T, label="目标 T", color=Figure.INK, r=5)
fig.show()

**做**。在 `vectors.py` 中实现：

- `linear_combination(coeffs, vectors)`：$\sum_i c_i v_i$，两个列表等长。
- `solve_2x2(v, w, t)`：求 (a, b) 使 $a v + b w = t$（均为二维）；v 与 w 共线时返回 `None`。提示：用消元法，或先算 $v_1 w_2 - v_2 w_1$ 判断是否为 0。

In [ ]:
status(
    linear_combination=([1, 1], [[1, 0], [0, 1]]),
    solve_2x2=([1, 0], [0, 1], [1, 1]),
)

In [ ]:
# 测
from vectors import linear_combination, solve_2x2

assert linear_combination([2, 1], [[3, 1], [1, 2]]) == [7, 4]
assert linear_combination([2, -3], [[1, 2], [3, -1]]) == [-7, 7]

a, b = solve_2x2([3, 1], [1, 2], [7, 4])
assert abs(a - 2) < 1e-9 and abs(b - 1) < 1e-9, "[7, 4] = 2·v + 1·w"

a, b = solve_2x2([1, 2], [2, -1], [8, 1])
assert abs(a - 2) < 1e-9 and abs(b - 3) < 1e-9, "[8, 1] = 2·v + 3·w"

assert solve_2x2([3, 1], [6, 2], [7, 4]) is None, "共线时应返回 None"
print("步骤 2 通过")

### 步骤 3 · Span：共线与不共线

**学**。Span 是全部 linear combination 的集合。u = (6, 2) 是 v 的 2 倍，任何组合仍是 v 的倍数，span 只是一条直线；w 不与 v 共线，两个系数各自独立，span 覆盖整个平面。

$$a \cdot v + b \cdot (2v) = (a + 2b) \cdot v$$

In [ ]:
# 玩：把系数 a、b 取遍 −2…2 的整数，标出全部 a·p + b·q 的落点
def show_span(p, q, q_name, title):
    """标出 a·p + b·q 在 a、b 取 −2…2 整数时的全部落点。"""
    fig = Figure(xlim=(-9, 9), ylim=(-6, 6), unit=30, title=title)
    for a in range(-2, 3):
        for b in range(-2, 3):
            fig.point((a * p[0] + b * q[0], a * p[1] + b * q[1]), color=Figure.GREY, r=3.5)
    fig.arrow(p, label="v", label_pos="mid", side=1)
    fig.arrow(q, label=q_name, color=Figure.BLUE, label_pos="mid", side=-1)
    fig.show()


show_span((3, 1), (6, 2), "u", "v 与 u = 2v 共线：落点排成一条直线")
show_span((3, 1), (1, 2), "w", "v 与 w 不共线：落点铺成覆盖平面的网格")

In [ ]:
# 玩：像素字母混合——25 维向量的 linear combination
import pixel_blend

letter_a = pixel_blend.to_vec(pixel_blend.A)   # 5×5 字母展开成 25 维向量
letter_b = pixel_blend.to_vec(pixel_blend.B)
for s in (0.0, 0.5, 1.0):
    mixed = pixel_blend.blend(letter_a, letter_b, s)   # (1 − s)·A + s·B
    print(f"s = {s}")
    print("\n".join(pixel_blend.render(mixed)))
    print()

**做**。在 `vectors.py` 中实现 `is_collinear(v, w, tol=1e-9)`：存在 k 使 w = k·v 或 v = k·w；零向量与任何向量共线。提示：二维时看 $v_1 w_2 - v_2 w_1$ 是否为 0；一般维数看所有分量对是否成比例。

In [ ]:
status(is_collinear=([1, 0], [2, 0]))

In [ ]:
# 测
from vectors import is_collinear

assert is_collinear([3, 1], [6, 2]) is True
assert is_collinear([2, -4], [-3, 6]) is True
assert is_collinear([3, 1], [1, 2]) is False
assert is_collinear([0, 0], [5, 7]) is True
print("步骤 3 通过")

### 步骤 4 · Linearly independent 与 basis

**学**。去掉某个 vector 而 span 不变，这组 vector 即 linearly dependent；每个 vector 都为 span 增加一个维度，则 linearly independent。既 linearly independent 又 span 整个空间的一组 vector 称为 basis，此时每个 vector 的坐标唯一。

$$(1, 1, 2) = 1 \cdot (1, 0, 1) + 1 \cdot (0, 1, 1)，\ 故\ v_1, v_2, v_3\ \text{linearly dependent}$$

换基：同一个 vector 在不同 basis 下坐标不同。[4, 2] 在标准基下坐标为 (4, 2)，在 {[1, 1], [1, −1]} 下坐标为 (3, 1)：$3 \cdot [1, 1] + 1 \cdot [1, -1] = [4, 2]$。

In [ ]:
# 玩：换基——同一个点，两组基下的坐标
from vectors import solve_2x2
target = [4, 2]
try:
    print("标准基 {[1,0],[0,1]}:", solve_2x2([1, 0], [0, 1], target))
    print("基 {[1,1],[1,-1]}   :", solve_2x2([1, 1], [1, -1], target))
except NotImplementedError:
    print("solve_2x2 尚未实现（步骤 2），实现后重新执行本单元格")

**做**。在 `vectors.py` 中实现：

- `coords_in_basis(t, b1, b2)`：t 在基 {b1, b2} 下的坐标；b1、b2 共线时抛出 `ValueError`。
- `in_span(t, v, w, tol=1e-9)`：三维，t 是否在不共线的 v、w 的 span 内。提示：用两个分量解出 a、b，再验证第三个分量。
- `is_independent_3(v1, v2, v3, tol=1e-9)`：三个三维向量是否 linearly independent。提示：v1、v2 不共线，且 v3 不在二者的 span 内。

In [ ]:
status(
    coords_in_basis=([1, 1], [1, 0], [0, 1]),
    in_span=([1, 1, 0], [1, 0, 0], [0, 1, 0]),
    is_independent_3=([1, 0, 0], [0, 1, 0], [0, 0, 1]),
)

In [ ]:
# 测
from vectors import coords_in_basis, in_span, is_independent_3

a, b = coords_in_basis([4, 2], [1, 1], [1, -1])
assert abs(a - 3) < 1e-9 and abs(b - 1) < 1e-9, "[4, 2] 在基 {[1, 1], [1, −1]} 下坐标为 (3, 1)"

try:
    coords_in_basis([1, 1], [1, 2], [2, 4])
    raise AssertionError("共线的基应抛出 ValueError")
except ValueError:
    pass

assert in_span([1, 2, 3], [1, 0, 1], [0, 1, 1]) is True
assert in_span([1, 2, 4], [1, 0, 1], [0, 1, 1]) is False
assert is_independent_3([1, 0, 1], [0, 1, 1], [1, 1, 2]) is False
assert is_independent_3([2, 0, 0], [0, 3, 0], [4, 6, 0]) is False
assert is_independent_3([1, 0, 0], [0, 1, 0], [0, 0, 1]) is True
print("步骤 4 通过")

## 工程应用 · Attention 的输出是 linear combination

**学**。Attention(Q, K, V) 的每一行是 V 各行的 linear combination，系数由 softmax 给出，非负且和为 1，因此输出落在 value vector 的 span 内，并且位于它们围成的区域之中。

示例：三个 value vector (1, 0)、(0, 1)、(2, 2)，权重 [0.7, 0.2, 0.1]，输出 $0.7 \cdot (1, 0) + 0.2 \cdot (0, 1) + 0.1 \cdot (2, 2) = (0.9, 0.4)$。

LoRA 同理：权重更新 $\Delta W = BA$，B 只有 r 列，更新矩阵的每一列都是这 r 个列 vector 的 linear combination，因此全部列位于维数不超过 r 的子空间内。

In [ ]:
# 玩：手调 score，看 softmax 权重与输出点在三角形内移动。改 scores 再执行
from attention_toy import VALUES, attention_output

for scores in ([0, 0, 0], [1.9459, 0.6931, 0.0], [0, 0, 8]):
    weights, out = attention_output(scores)
    rounded = [round(w, 2) for w in weights]
    fig = Figure(xlim=(-1, 3), ylim=(-1, 3), unit=60, title=f"score = {scores} → 权重 {rounded}")
    fig.polygon(list(VALUES.values()))   # 三个 value vector 围成的三角形；输出只能落在其中
    for name, value in VALUES.items():
        fig.arrow(value, label=f"{name} = {value}")
    fig.point(out, label=f"输出 ({out[0]:.2f}, {out[1]:.2f})", color=Figure.BLUE, r=6)
    fig.show()

**做**。在 `vectors.py` 中实现 `weighted_sum(weights, vectors)`：权重非负且和为 1（容差 1e-9）时返回 `linear_combination(weights, vectors)`，否则抛出 `ValueError`。

In [ ]:
status(weighted_sum=([1.0], [[1, 0]]))

In [ ]:
# 测
from vectors import weighted_sum

out = weighted_sum([0.7, 0.2, 0.1], [[1, 0], [0, 1], [2, 2]])
assert all(abs(x - y) < 1e-9 for x, y in zip(out, [0.9, 0.4])), "输出应为 (0.9, 0.4)"

for bad in ([0.5, 0.7], [1.5, -0.5]):   # 和不为 1；有负权重
    try:
        weighted_sum(bad, [[1, 0], [0, 1]])
        raise AssertionError(f"权重 {bad} 应抛出 ValueError")
    except ValueError:
        pass
print("工程应用通过")

## 总验证与测验

全部实现后运行总验证器（23 个用例 + 装有 numpy 时 200 组随机对拍）；然后做题库中本篇的选择题。

```bash
make test-linalg     # 在仓库根目录
```

In [ ]:
# 总验证：与 make test-linalg 相同
import subprocess, sys
r = subprocess.run([sys.executable, "test_vectors.py"], capture_output=True, text=True)
print(r.stdout[-1500:]); print("RESULT PASS" if r.returncode == 0 else "RESULT FAIL（骨架状态下为预期）")

## 适用条件与局限

本 notebook 只处理二维、三维与实数 scalar。Basis 不唯一；判断多个 vector 是否 linearly independent，一般需要计算 rank 或 determinant（第 04 篇）。演示用字符画板，不依赖绘图库；需要精确图形时用 numpy 与 matplotlib 另行绘制。